# Reproduce the numbers, Table 1 and Fig. 1 of the paper

*When misclassified is not mislabeled: a controlled diagnosis of systematic errors in positive-unlabeled customer data*
(ESANN 2027, submitted).

The customer data are confidential. This notebook works only on the aggregated results in `paper/results/`
(one folder per evaluation run, CSV). It

1. recomputes every number in the paper from these files, prints it next to the value in the paper and stops if a
   number differs (`paper/paper_numbers.py`, the same table as `docs/numbers.md`),
2. rebuilds Table 1 from the same numbers and checks it row by row against the paper,
3. draws Fig. 1 in its final size (`output/fig1.pdf`, `output/fig1.png`).

It runs in a few seconds.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "paper").exists() else ROOT.parent      # works from the repository root or from paper/
sys.path[:0] = [str(ROOT / "paper"), str(ROOT / "paper" / "figures")]
import pandas as pd
import paper_numbers as PN

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

## 1. Every number in the paper

In [ ]:
N = PN.entries()
N["same"] = N.paper == N.computed
num = N[~N.quantity.isin(PN.VERBAL)]
print(f"{int(num.same.sum())} of {len(num)} numbers print exactly as in the paper; "
      f"{len(N) - len(num)} entries are statements or external numbers (values behind them below).")
assert num.same.all(), num.loc[~num.same, ["quantity", "paper", "computed"]]
N[["location", "quantity", "paper", "computed", "file", "same"]]

In [ ]:
N.loc[N.quantity.isin(PN.VERBAL), ["location", "quantity", "paper", "computed"]]

In [ ]:
print(PN.NOTES)

## 2. Table 1 (diagnosis protocol)

In [ ]:
T1 = PN.table1(N)
paper_T1 = pd.DataFrame(PN.TABLE1, columns=T1.columns)
same = (T1 == paper_T1).all(axis=1)
print(f"{int(same.sum())} of {len(T1)} rows of Table 1 equal the paper")
assert same.all(), T1[~same]
print(T1.to_string(index=False))

## 3. Fig. 1

Same code as the figure in the paper (`paper/figures/make_fig1.py`), width 12.2 cm (text width of `esannV2.cls`),
height 4 cm. The PNG is pixel-identical to the PNG of the submitted figure.

In [ ]:
import make_fig1
from IPython.display import Image, display

make_fig1.make(ROOT / "output")
display(Image(filename=str(ROOT / "output" / "fig1.png"), width=700))

## 4. What cannot be recomputed here

The model fits, the out-of-fold predictions and all object-level quantities (errors per customer, candidate
lists, twin pairs, the website check) need the confidential customer data and are not part of this repository.
The enrichment pipeline and the LLM agent are proprietary; for the website check, only aggregated counts are given.
The reusable package follows after the conference submission.